# 00 - Speaker Diarization

Identifies which segments belong to which speaker (male vs female) using **pyannote.audio**.
Uses voice embeddings to ensure consistent speaker labels across all videos.

**Requirements:**
- Google Colab with GPU
- Free HuggingFace token (get one at https://huggingface.co/settings/tokens)
- Accept model conditions at https://huggingface.co/pyannote/speaker-diarization-community-1

**Output:** Updated translation JSONs with a `speaker` field per segment.

In [ ]:
!pip install -q pyannote.audio
!pip install numpy==2.1.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/AIVideoLanguageTransformation'
AUDIO_DIR = os.path.join(PROJECT_DIR, 'data', 'audio', 'original')
TRANSLATIONS_DIR = os.path.join(PROJECT_DIR, 'data', 'translations')

print(f'Audio files: {sorted(os.listdir(AUDIO_DIR))}')
print(f'Translation files: {sorted(os.listdir(TRANSLATIONS_DIR))}')

In [ ]:
# Set your HuggingFace token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('Loaded HF_TOKEN from Colab Secrets')
except:
    HF_TOKEN = ''  # Paste your token here if not using Colab Secrets
    print('Using hardcoded HF_TOKEN')

In [ ]:
import torch
from pyannote.audio import Pipeline, Inference

print('Loading pyannote diarization pipeline...')
pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=HF_TOKEN
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pipeline.to(device)

# Load embedding model for cross-video speaker matching
embedding_model = Inference(
    'pyannote/embedding',
    token=HF_TOKEN,
    window='whole'
)
embedding_model.to(device)

print(f'Pipeline loaded on {device}.')

In [ ]:
import json
import numpy as np
from pydub import AudioSegment
from collections import defaultdict
import tempfile


def diarize_audio(audio_path, num_speakers=2):
    """Run speaker diarization and return speaker segments."""
    print(f'Diarizing: {os.path.basename(audio_path)}')
    output = pipeline(audio_path, num_speakers=num_speakers)

    speaker_segments = []
    for turn, _, speaker in output.speaker_diarization.itertracks(yield_label=True):
        speaker_segments.append({
            'start': turn.start,
            'end': turn.end,
            'speaker': speaker
        })

    # Print summary
    speakers = defaultdict(float)
    for s in speaker_segments:
        speakers[s['speaker']] += s['end'] - s['start']
    for sp, dur in speakers.items():
        print(f'  {sp}: {dur:.1f}s')

    return speaker_segments


def get_speaker_embedding(audio_path, speaker_segments, speaker_label):
    """Extract average voice embedding for a specific speaker."""
    audio = AudioSegment.from_wav(audio_path)
    # Collect longest segments for this speaker
    segs = [(s['end'] - s['start'], s) for s in speaker_segments if s['speaker'] == speaker_label]
    segs.sort(key=lambda x: -x[0])

    combined = AudioSegment.empty()
    total = 0
    for dur, seg in segs:
        if total >= 15:  # 15s is enough for embedding
            break
        start_ms = int(seg['start'] * 1000)
        end_ms = int(seg['end'] * 1000)
        combined += audio[start_ms:end_ms]
        total += dur

    # Save temp file and compute embedding
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        combined.export(f.name, format='wav')
        embedding = embedding_model(f.name)
    os.unlink(f.name)
    return embedding


def assign_speakers(segments, speaker_segments):
    """Assign a speaker label to each transcript segment based on max overlap."""
    for seg in segments:
        best_speaker = 'UNKNOWN'
        best_overlap = 0.0
        for sp in speaker_segments:
            overlap_start = max(seg['start'], sp['start'])
            overlap_end = min(seg['end'], sp['end'])
            overlap = max(0.0, overlap_end - overlap_start)
            if overlap > best_overlap:
                best_overlap = overlap
                best_speaker = sp['speaker']
        seg['speaker'] = best_speaker
    return segments

In [ ]:
from scipy.spatial.distance import cosine

audio_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith('.wav')])

# Step 1: Diarize all videos and collect embeddings per speaker
all_results = {}  # {video_stem: {speaker_segments, embeddings}}

for audio_file in audio_files:
    audio_path = os.path.join(AUDIO_DIR, audio_file)
    stem = os.path.splitext(audio_file)[0]
    trans_path = os.path.join(TRANSLATIONS_DIR, f'{stem}_en.json')
    if not os.path.exists(trans_path):
        print(f'No translation file for {stem}, skipping')
        continue

    print(f'\n{"="*60}')
    print(f'Processing: {audio_file}')

    speaker_segments = diarize_audio(audio_path, num_speakers=2)

    # Get unique speakers and their embeddings
    unique_speakers = set(s['speaker'] for s in speaker_segments)
    embeddings = {}
    for sp in unique_speakers:
        embeddings[sp] = get_speaker_embedding(audio_path, speaker_segments, sp)
        print(f'  Computed embedding for {sp}')

    all_results[stem] = {
        'speaker_segments': speaker_segments,
        'embeddings': embeddings,
        'audio_path': audio_path,
        'trans_path': trans_path,
    }

print('\nDiarization complete for all videos.')

In [ ]:
# Step 2: Match speakers across videos using embedding similarity
# Use the first video as the reference for speaker identity

stems = sorted(all_results.keys())
ref_stem = stems[0]
ref_embeddings = all_results[ref_stem]['embeddings']
ref_speakers = sorted(ref_embeddings.keys())  # e.g., ['SPEAKER_00', 'SPEAKER_01']

print(f'Reference video: {ref_stem}')
print(f'Reference speakers: {ref_speakers}')

# Build speaker mapping for each video
speaker_mappings = {ref_stem: {sp: sp for sp in ref_speakers}}  # identity for reference

for stem in stems[1:]:
    embeddings = all_results[stem]['embeddings']
    video_speakers = sorted(embeddings.keys())
    mapping = {}

    print(f'\nMatching speakers for {stem}:')
    for v_sp in video_speakers:
        # Find closest reference speaker by cosine similarity
        best_ref = None
        best_sim = -1
        for r_sp in ref_speakers:
            sim = 1 - cosine(embeddings[v_sp], ref_embeddings[r_sp])
            print(f'  {v_sp} vs ref {r_sp}: similarity = {sim:.3f}')
            if sim > best_sim:
                best_sim = sim
                best_ref = r_sp
        mapping[v_sp] = best_ref
        print(f'  → {v_sp} mapped to {best_ref} (sim={best_sim:.3f})')

    # Check for conflicts (both mapped to same speaker)
    if len(set(mapping.values())) < len(mapping):
        print(f'  WARNING: Speaker mapping conflict! Resolving by best match...')
        # Use Hungarian algorithm-like approach: assign greedily by best similarity
        used = set()
        sorted_pairs = []
        for v_sp in video_speakers:
            for r_sp in ref_speakers:
                sim = 1 - cosine(embeddings[v_sp], ref_embeddings[r_sp])
                sorted_pairs.append((sim, v_sp, r_sp))
        sorted_pairs.sort(reverse=True)
        mapping = {}
        used_refs = set()
        for sim, v_sp, r_sp in sorted_pairs:
            if v_sp not in mapping and r_sp not in used_refs:
                mapping[v_sp] = r_sp
                used_refs.add(r_sp)

    speaker_mappings[stem] = mapping
    print(f'  Final mapping: {mapping}')

print(f'\nAll speaker mappings: {speaker_mappings}')

In [ ]:
# Step 3: Apply consistent speaker labels to all translation files

for stem, data in all_results.items():
    mapping = speaker_mappings[stem]
    speaker_segments = data['speaker_segments']

    # Remap speaker labels
    for s in speaker_segments:
        s['speaker'] = mapping.get(s['speaker'], s['speaker'])

    # Load translations and assign speakers
    with open(data['trans_path'], 'r', encoding='utf-8') as f:
        segments = json.load(f)

    segments = assign_speakers(segments, speaker_segments)

    with open(data['trans_path'], 'w', encoding='utf-8') as f:
        json.dump(segments, f, ensure_ascii=False, indent=2)

    # Summary
    speakers = defaultdict(int)
    for seg in segments:
        speakers[seg.get('speaker', 'UNKNOWN')] += 1
    print(f'{stem}: {dict(speakers)}')

print('\nAll translations updated with consistent speaker labels.')

In [ ]:
# Step 4: Extract voice references per speaker (from first video)

VOICE_REF_DIR = os.path.join(PROJECT_DIR, 'data', 'audio', 'voice_reference')
os.makedirs(VOICE_REF_DIR, exist_ok=True)

ref_data = all_results[ref_stem]
audio = AudioSegment.from_wav(ref_data['audio_path'])

with open(ref_data['trans_path'], 'r', encoding='utf-8') as f:
    segments = json.load(f)

speaker_segs = defaultdict(list)
for seg in segments:
    sp = seg.get('speaker', 'UNKNOWN')
    duration = seg['end'] - seg['start']
    speaker_segs[sp].append((duration, seg))

for speaker, segs in speaker_segs.items():
    segs.sort(key=lambda x: -x[0])  # longest first
    combined = AudioSegment.empty()
    total = 0
    for dur, seg in segs:
        if total >= 30:
            break
        start_ms = int(seg['start'] * 1000)
        end_ms = int(seg['end'] * 1000)
        combined += audio[start_ms:end_ms]
        total += dur

    ref_path = os.path.join(VOICE_REF_DIR, f'{speaker}_ref.wav')
    combined.export(ref_path, format='wav')
    print(f'Saved: {ref_path} ({total:.1f}s)')

print(f'\nVoice references saved to {VOICE_REF_DIR}')
print('Listen to each to verify which is male and which is female.')

In [ ]:
# Preview: show first 10 segments with speaker labels for each video
for trans_file in sorted(os.listdir(TRANSLATIONS_DIR)):
    if not trans_file.endswith('_en.json'):
        continue
    with open(os.path.join(TRANSLATIONS_DIR, trans_file), 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'\n=== {trans_file} ===')
    for seg in data[:10]:
        sp = seg.get('speaker', '?')
        print(f"  [{sp}] [{seg['start']:.1f}s-{seg['end']:.1f}s] {seg.get('text_en', seg.get('text_en_deepl', ''))}")